In [ ]:

# ============================================================
# Customer Sentiment Analysis Prediction
# Olist Brazilian E-Commerce Dataset
# ============================================================
# Pipeline: Preprocessing → EDA → Encoding → Scaling →
#           Outlier Removal → Class Balancing → Model Training → Pickling
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from scipy.stats import zscore

# XGBoost
from xgboost import XGBClassifier

# For class imbalance
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# For pickling
import pickle

# ============================================================
# 1. LOAD DATA
# ============================================================
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)

df = pd.read_csv('/content/drive/MyDrive/ExecutiveProgramInAdvanced-AI ML-IIT Palakkad-Feb2026/olist_master_dataset.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

# ============================================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
print("\n" + "=" * 60)
print("2. EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 60)

# Basic info
print("\n--- Dataset Info ---")
print(df.info())

# Statistical summary
print("\n--- Numerical Features Summary ---")
print(df.describe())

# Target variable distribution
print("\n--- Target Variable: review_score Distribution ---")
print(df['review_score'].value_counts().sort_index())
print(f"\nPercentage distribution:")
print((df['review_score'].value_counts(normalize=True).sort_index() * 100).round(2))

# Check class imbalance
print("\n--- Class Imbalance Check ---")
total = df['review_score'].count()
for score in sorted(df['review_score'].dropna().unique()):
    count = (df['review_score'] == score).sum()
    print(f"  Score {int(score)}: {count} ({count/total*100:.1f}%)")

# Categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"\n--- Categorical Columns ({len(cat_cols)}): ---")
print(cat_cols)
print(f"\n--- Numerical Columns ({len(num_cols)}): ---")
print(num_cols)

# Missing values analysis
print("\n--- Missing Values ---")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
print(missing_df[missing_df['Missing_Count'] > 0])

# Correlation for numerical features
print("\n--- Correlation with review_score (top features) ---")
num_df = df[num_cols].dropna()
if 'review_score' in num_df.columns:
    corr = num_df.corr()['review_score'].drop('review_score').abs().sort_values(ascending=False)
    print(corr.head(10))

# ============================================================
# 3. DATA PREPROCESSING
# ============================================================
print("\n" + "=" * 60)
print("3. DATA PREPROCESSING")
print("=" * 60)

# 3a. Drop unnecessary ID columns and text columns
drop_cols = ['order_id', 'customer_id', 'customer_unique_id', 'product_id',
             'seller_id', 'review_id', 'review_comment_title',
             'review_comment_message']
df_clean = df.drop(columns=drop_cols, errors='ignore')
print(f"Dropped ID/text columns: {drop_cols}")


# List of datetime columns to convert
datetime_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp'
]

# Convert each column to datetime (coerce errors to NaT for invalid entries)
for col in datetime_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')


# 3b. Feature Engineering from datetime columns
print("\n--- Feature Engineering from Dates ---")
# Calculate delivery time (days)
df_clean['delivery_days'] = (df_clean['order_delivered_customer_date'] -
                              df_clean['order_purchase_timestamp']).dt.days

# Calculate if delivery was late (vs estimated)
df_clean['delivery_vs_estimated'] = (df_clean['order_estimated_delivery_date'] -
                                      df_clean['order_delivered_customer_date']).dt.days

# Calculate approval time (hours)
df_clean['approval_hours'] = (df_clean['order_approved_at'] -
                               df_clean['order_purchase_timestamp']).dt.total_seconds() / 3600

# Drop original datetime columns
datetime_cols = df_clean.select_dtypes(include=['datetime64']).columns.tolist()
df_clean = df_clean.drop(columns=datetime_cols)
print(f"Created: delivery_days, delivery_vs_estimated, approval_hours")
print(f"Dropped datetime columns: {datetime_cols}")

# 3c. Handle Missing Values
print("\n--- Handling Missing Values ---")
print(f"Missing values before: {df_clean.isnull().sum().sum()}")

# Drop rows where target is missing
df_clean = df_clean.dropna(subset=['review_score'])

# For numerical columns - fill with median
num_cols_clean = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols_clean.remove('review_score')  # Don't fill target
for col in num_cols_clean:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# For categorical columns - fill with mode
cat_cols_clean = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols_clean:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f"Missing values after: {df_clean.isnull().sum().sum()}")
print(f"Dataset shape after missing value removal: {df_clean.shape}")

# ============================================================
# 4. LABEL ENCODING FOR CATEGORICAL COLUMNS
# ============================================================
print("\n" + "=" * 60)
print("4. LABEL ENCODING FOR CATEGORICAL COLUMNS")
print("=" * 60)

cat_cols_final = df_clean.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in cat_cols_final:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded '{col}': {len(le.classes_)} unique values")

print(f"\nDataset shape after encoding: {df_clean.shape}")
print(f"All columns now numeric: {df_clean.dtypes.value_counts().to_dict()}")

# ============================================================
# 5. OUTLIER REMOVAL USING IQR METHOD
# ============================================================
print("\n" + "=" * 60)
print("5. OUTLIER REMOVAL (IQR Method)")
print("=" * 60)

# Select numerical features for outlier detection (exclude target & encoded categoricals)
outlier_cols = ['price', 'freight_value', 'product_weight_g', 'product_length_cm',
                'product_height_cm', 'product_width_cm', 'payment_value',
                'product_name_lenght', 'product_description_lenght']

# Filter to only columns that exist
outlier_cols = [col for col in outlier_cols if col in df_clean.columns]

print(f"Checking outliers in: {outlier_cols}")
print(f"Shape before outlier removal: {df_clean.shape}")

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    if outliers > 0:
        print(f"  {col}: {outliers} outliers detected")

    # Remove outliers
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"Shape after outlier removal: {df_clean.shape}")

# ============================================================
# 6. FEATURE SCALING (StandardScaler)
# ============================================================
print("\n" + "=" * 60)
print("6. FEATURE SCALING (StandardScaler)")
print("=" * 60)

# Separate features and target
X = df_clean.drop(columns=['review_score'])
y = df_clean['review_score'].astype(int)

# Scale numerical features
scaler = StandardScaler()
feature_names = X.columns.tolist()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names, index=X.index)

print(f"Features shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}")
print(f"\nScaled features sample (first 3 rows):")
print(X_scaled.head(3))

# ============================================================
# 7. HANDLING CLASS IMBALANCE (SMOTE + Undersampling)
# ============================================================
print("\n" + "=" * 60)
print("7. HANDLING CLASS IMBALANCE")
print("=" * 60)

print(f"\nOriginal class distribution:")
print(Counter(y))

# Strategy: Combine SMOTE (oversampling minority) + Random Undersampling (majority)
# SMOTE to oversample minority classes
smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=1)

# Apply SMOTE
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print(f"\nAfter SMOTE resampling:")
print(Counter(y_resampled))
print(f"Resampled dataset size: {X_resampled.shape[0]}")

# ============================================================
# 8. TRAIN-TEST SPLIT
# ============================================================
print("\n" + "=" * 60)
print("8. TRAIN-TEST SPLIT (80-20)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution: {Counter(y_train)}")
print(f"Testing class distribution: {Counter(y_test)}")

# ============================================================
# 9. MODEL 1: RANDOM FOREST + RandomizedSearchCV (OPTIMIZED)
# ============================================================
print("\n" + "=" * 60)
print("9. MODEL 1: RANDOM FOREST + RandomizedSearchCV (OPTIMIZED)")
print("=" * 60)

# Reduced & focused hyperparameter grid
rf_param_dist = {
    'n_estimators': [50, 100, 150],         # Reduced from [50-500]
    'max_depth': [5, 10, 15, 20],           # Removed None (unlimited = very slow)
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],       # Removed None (all features = slow)
    'criterion': ['gini', 'entropy']
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Reduced iterations and folds
rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_dist,
    n_iter=20,              # Reduced from 50
    cv=3,                   # Reduced from 5
    scoring='accuracy',
    random_state=42,
    n_jobs=1,               # Avoid nested parallelism issues
    verbose=2
)

print("Training Random Forest with Hyperparameter Tuning...")
print(f"Total fits: {20 * 3} = 60 (reduced from 250)")
rf_random_search.fit(X_train, y_train)

print(f"\nBest Parameters: {rf_random_search.best_params_}")
print(f"Best CV Score: {rf_random_search.best_score_:.4f}")

# Predictions
rf_best = rf_random_search.best_estimator_
y_pred_rf = rf_best.predict(X_test)

print(f"\n--- Random Forest Results ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# ============================================================
# 10. MODEL 2: DECISION TREE CLASSIFIER (OPTIMIZED)
# ============================================================
print("\n" + "=" * 60)
print("10. MODEL 2: DECISION TREE CLASSIFIER (OPTIMIZED)")
print("=" * 60)

dt_param_dist = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'splitter': ['best']                   # Removed 'random' - slower
}

dt_model = DecisionTreeClassifier(random_state=42)

dt_random_search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=dt_param_dist,
    n_iter=15,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Training Decision Tree...")
dt_random_search.fit(X_train, y_train)

print(f"\nBest Parameters: {dt_random_search.best_params_}")
print(f"Best CV Score: {dt_random_search.best_score_:.4f}")

dt_best = dt_random_search.best_estimator_
y_pred_dt = dt_best.predict(X_test)

print(f"\n--- Decision Tree Results ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dt))




1. LOADING DATA
Dataset Shape: (119143, 37)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'product_category_name_english']

First 5 rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a9

In [ ]:

# ============================================================
# 11. MODEL 3: XGBOOST CLASSIFIER
# ============================================================
print("\n" + "=" * 60)
print("11. MODEL 3: XGBOOST CLASSIFIER")
print("=" * 60)

xgb_param_dist = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
}


# Shift labels from [1,2,3,4,5] → [0,1,2,3,4]

y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=1,
    verbose=2
)

print("Training XGBoost...")
xgb_random_search.fit(X_train, y_train_xgb)  # Use shifted labels

print(f"\nBest Parameters: {xgb_random_search.best_params_}")
print(f"Best CV Score: {xgb_random_search.best_score_:.4f}")

xgb_best = xgb_random_search.best_estimator_
y_pred_xgb_shifted = xgb_best.predict(X_test)


# Shift predictions back from [0,1,2,3,4] → [1,2,3,4,5]

y_pred_xgb = y_pred_xgb_shifted + 1

print(f"\n--- XGBoost Results ---")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))


# ============================================================
# 12. MODEL COMPARISON & TESTING ON SAMPLE DATA
# ============================================================
print("\n" + "=" * 60)
print("12. MODEL COMPARISON & TESTING ON SAMPLE DATA")
print("=" * 60)

# --- Accuracy Comparison ---
results = {
    'Random Forest': accuracy_score(y_test, y_pred_rf),
    'Decision Tree': accuracy_score(y_test, y_pred_dt),
    'XGBoost': accuracy_score(y_test, y_pred_xgb)
}


print("     MODEL ACCURACY COMPARISON  ")

for model_name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(acc * 30)
    print(f"║  {model_name:<16} : {acc*100:.2f}% {bar}")

best_model_name = max(results, key=results.get)
print(f"\n Best Model: {best_model_name} with accuracy {results[best_model_name]*100:.2f}%")

# --- Detailed Metrics Comparison ---
print("\n--- Detailed Metrics (Weighted Avg) ---")
from sklearn.metrics import precision_score, recall_score, f1_score

metrics_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Decision Tree', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_rf, average='weighted', zero_division=0),
        precision_score(y_test, y_pred_dt, average='weighted', zero_division=0),
        precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, y_pred_rf, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_dt, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_rf, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_dt, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    ]
})

metrics_comparison = metrics_comparison.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(metrics_comparison.to_string(index=False))

# --- Testing on Sample Test Data ---
print("\n" + "-" * 60)
print("TESTING ON SAMPLE TEST DATA (10 random samples)")
print("-" * 60)

# Select random samples from test set
n_samples = min(10, len(X_test))
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=n_samples, replace=False)

# Get sample data
if hasattr(X_test, 'iloc'):
    X_sample = X_test.iloc[sample_indices]
    y_sample_actual = y_test.iloc[sample_indices]
else:
    X_sample = X_test[sample_indices]
    y_sample_actual = y_test[sample_indices]

# Get predictions from all 3 models
rf_sample_preds = rf_best.predict(X_sample)
dt_sample_preds = dt_best.predict(X_sample)
xgb_sample_preds = xgb_best.predict(X_sample)

# Create a comparison dataframe
sample_results = pd.DataFrame({
    'Sample #': range(1, n_samples + 1),
    'Actual': y_sample_actual.values,
    'RF_Pred': rf_sample_preds,
    'DT_Pred': dt_sample_preds,
    'XGB_Pred': xgb_sample_preds
})

# Add correctness flags
sample_results['RF_Correct'] = sample_results['Actual'] == sample_results['RF_Pred']
sample_results['DT_Correct'] = sample_results['Actual'] == sample_results['DT_Pred']
sample_results['XGB_Correct'] = sample_results['Actual'] == sample_results['XGB_Pred']

print("\n" + f"{'Sample':<8} {'Actual':<8} {'RF Pred':<9} {'DT Pred':<9} {'XGB Pred':<9} {'RF ✓':<6} {'DT ✓':<6} {'XGB ✓':<6}")
print("=" * 70)
for _, row in sample_results.iterrows():
    rf_mark = "✅" if row['RF_Correct'] else "❌"
    dt_mark = "✅" if row['DT_Correct'] else "❌"
    xgb_mark = "✅" if row['XGB_Correct'] else "❌"
    print(f"  {int(row['Sample #']):<6} {int(row['Actual']):<8} {int(row['RF_Pred']):<9} {int(row['DT_Pred']):<9} {int(row['XGB_Pred']):<9} {rf_mark:<6} {dt_mark:<6} {xgb_mark:<6}")

# --- Summary of Sample Predictions ---
print("\n--- Sample Prediction Accuracy ---")
rf_sample_acc = sample_results['RF_Correct'].sum() / n_samples * 100
dt_sample_acc = sample_results['DT_Correct'].sum() / n_samples * 100
xgb_sample_acc = sample_results['XGB_Correct'].sum() / n_samples * 100

print(f"  Random Forest : {int(sample_results['RF_Correct'].sum())}/{n_samples} correct ({rf_sample_acc:.1f}%)")
print(f"  Decision Tree : {int(sample_results['DT_Correct'].sum())}/{n_samples} correct ({dt_sample_acc:.1f}%)")
print(f"  XGBoost       : {int(sample_results['XGB_Correct'].sum())}/{n_samples} correct ({xgb_sample_acc:.1f}%)")

# --- Confusion Matrices ---
print("\n--- Confusion Matrices ---")
print(f"\nRandom Forest:")
print(confusion_matrix(y_test, y_pred_rf))

print(f"\nDecision Tree:")
print(confusion_matrix(y_test, y_pred_dt))

print(f"\nXGBoost:")
print(confusion_matrix(y_test, y_pred_xgb))

# --- Feature Importance (from best model) ---
print("\n--- Top 10 Feature Importances (Best Model: {}) ---".format(best_model_name))
best_models_dict = {
    'Random Forest': rf_best,
    'Decision Tree': dt_best,
    'XGBoost': xgb_best
}
best = best_models_dict[best_model_name]

if hasattr(best, 'feature_importances_'):
    feature_imp = pd.DataFrame({
        'Feature': X_train.columns if hasattr(X_train, 'columns') else [f'feature_{i}' for i in range(X_train.shape[1])],
        'Importance': best.feature_importances_
    }).sort_values('Importance', ascending=False).head(10)

    print(feature_imp.to_string(index=False))

# ============================================================
# 13. PICKLE THE BEST MODELS
# ============================================================
print("\n" + "=" * 60)
print("13. PICKLING THE BEST MODELS")
print("=" * 60)

import pickle

models_to_save = {
    'random_forest_best.pkl': rf_best,
    'decision_tree_best.pkl': dt_best,
    'xgboost_best.pkl': xgb_best,
    'scaler.pkl': scaler,
    'label_encoders.pkl': label_encoders
}

for filename, model in models_to_save.items():
    with open(filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"  ✅ Saved: {filename}")

# Save the best overall model
best_overall = best_models_dict[best_model_name]
with open('best_sentiment_model.pkl', 'wb') as f:
    pickle.dump(best_overall, f)
print(f"\n   Best overall model saved as: best_sentiment_model.pkl ({best_model_name})")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print(" PIPELINE COMPLETE - FINAL SUMMARY")
print("=" * 60)
print(f"""
  Best Model        : {best_model_name}
  Best Accuracy     : {results[best_model_name]*100:.2f}%
  Sample Test Acc   : RF={rf_sample_acc:.1f}%, DT={dt_sample_acc:.1f}%, XGB={xgb_sample_acc:.1f}%
  Models Pickled    : 5 files saved (.pkl)

  Files Created:
    - random_forest_best.pkl
    - decision_tree_best.pkl
    - xgboost_best.pkl
    - scaler.pkl
    - label_encoders.pkl
    - best_sentiment_model.pkl
""")



11. MODEL 3: XGBOOST CLASSIFIER
Training XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, min_child_weight=3, n_estimators=50, subsample=0.7; total time=   6.1s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, min_child_weight=3, n_estimators=50, subsample=0.7; total time=   4.0s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, min_child_weight=3, n_estimators=50, subsample=0.7; total time=   4.0s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=0.8; total time=   9.2s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=0.8; total time=   9.2s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=7, min_child_weight=3, n_estimators=100, subsample=0.8; total time=   7.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=7, min_child_weigh